# ASTER: Attentional Scene Text Recognition - Complete Tutorial

This notebook provides a comprehensive walkthrough of the ASTER (Attentional Scene Text Recognition) implementation.

## What is ASTER?

ASTER is an end-to-end scene text recognition system that:
1. **Rectifies** curved/distorted text using Thin Plate Spline (TPS) transformation
2. **Extracts features** using a CNN (ResNet-based)
3. **Encodes sequences** using Bidirectional LSTM
4. **Decodes** text using an attention mechanism

## Architecture Overview

```
Input Image (Curved Text)
    ↓
TPS Rectification Network
    ↓
Rectified Image (Straight Text)
    ↓
CNN Feature Extractor (ResNet)
    ↓
Feature Maps
    ↓
Bidirectional LSTM
    ↓
Sequential Features
    ↓
Attention Decoder
    ↓
Predicted Text
```

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import sys

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Step 1: Understanding the Configuration

First, let's look at the configuration class that controls all hyperparameters.

In [ ]:
# Import configuration
from config import get_config

# Get configuration for Synth90K dataset
config = get_config('Synth90K')

print("ASTER Configuration:")
print("=" * 50)
print(f"Image size: {config.IMG_HEIGHT} x {config.IMG_WIDTH}")
print(f"Number of fiducial points: {config.NUM_FIDUCIAL}")
print(f"Hidden size (LSTM): {config.HIDDEN_SIZE}")
print(f"Attention dimension: {config.ATTENTION_DIM}")
print(f"Embedding dimension: {config.EMBEDDING_DIM}")
print(f"Max sequence length: {config.MAX_SEQ_LENGTH}")
print(f"Batch size: {config.BATCH_SIZE}")
print(f"Learning rate: {config.LEARNING_RATE}")
print(f"")
print(f"Character set length: {config.NUM_CLASSES}")
print(f"Characters: {config.CHARACTERS[:50]}...")

## Step 2: TPS Rectification Network

The TPS (Thin Plate Spline) rectification network transforms curved or distorted text into horizontal text.

### Components:
1. **Localization Network**: Predicts control points from the input image
2. **Grid Generator**: Creates a sampling grid using TPS transformation
3. **Sampler**: Applies the grid to rectify the image

In [ ]:
from rectification import TPSRectification, LocalizationNetwork, GridGenerator

# Create a sample curved text image (simulated)
B, C, H, W = 2, 3, 32, 100
sample_image = torch.randn(B, C, H, W)

print("Step 2: TPS Rectification Network")
print("=" * 50)
print(f"Input shape: {sample_image.shape}")

# Create rectification module
rectifier = TPSRectification(
    num_fiducial=config.NUM_FIDUCIAL,
    img_height=H,
    img_width=W,
    num_channels=C
)

# Forward pass
rectified, ctrl_points = rectifier(sample_image)

print(f"\nRectified shape: {rectified.shape}")
print(f"Control points shape: {ctrl_points.shape}")
print(f"Control points range: [{ctrl_points.min():.4f}, {ctrl_points.max():.4f}]")

## Step 3: CNN Feature Extractor

The CNN extracts visual features from the rectified text image. We use a ResNet-like architecture.

### Architecture:
- Conv layers with batch normalization
- Residual blocks
- Downsampling via max pooling

Input: (B, 3, 32, 100)
Output: (B, 512, H/16, W/16) ≈ (B, 512, 2, 7)

In [ ]:
from feature_extractor import ResNetFeatureExtractor, CNNFeatureExtractor

print("\nStep 3: CNN Feature Extractor")
print("=" * 50)

# Create ResNet feature extractor
resnet_extractor = ResNetFeatureExtractor(
    num_channels=3,
    output_channels=512
)

# Extract features from rectified image
features = resnet_extractor(rectified)

print(f"Input (rectified): {rectified.shape}")
print(f"Output features: {features.shape}")
print(f"Feature channels: {features.size(1)}")
print(f"Spatial dimensions: {features.size(2)} x {features.size(3)}")

# Let's also try the VGG-style extractor
vgg_extractor = CNNFeatureExtractor(num_channels=3, num_filters=512)
vgg_features = vgg_extractor(rectified)
print(f"\nVGG-style features: {vgg_features.shape}")

## Step 4: Bidirectional LSTM

The BiLSTM models sequential dependencies in the extracted features.

### What it does:
- Takes CNN features averaged over height: (B, C, W)
- Processes left-to-right and right-to-left
- Outputs context-aware sequence representations: (B, W, hidden_size*2)

In [ ]:
from bidirectional_lstm import SequenceEncoder, BidirectionalLSTM

print("\nStep 4: Bidirectional LSTM")
print("=" * 50)

# Create sequence encoder
sequence_encoder = SequenceEncoder(
    input_channels=512,
    hidden_size=config.HIDDEN_SIZE,
    num_layers=2
)

# Encode features
sequence_features = sequence_encoder(features)

print(f"Input features: {features.shape}")
print(f"Output sequences: {sequence_features.shape}")
print(f"Sequence length: {sequence_features.size(1)}")
print(f"Feature dimension: {sequence_features.size(2)}")
print(f"(hidden_size * 2 = {config.HIDDEN_SIZE} * 2 = {config.HIDDEN_SIZE * 2})")

## Step 5: Attention Decoder

The attention decoder generates text character by character, attending to relevant parts of the sequence.

### Architecture:
1. **Embedding Layer**: Converts character indices to dense vectors
2. **Attention Mechanism**: Computes attention weights over encoder features
3. **Dual LSTM**: Two LSTMs - one for attention, one for character prediction
4. **Output Projection**: Maps LSTM output to character probabilities

In [ ]:
from attention_decoder import AttentionDecoderV2

print("\nStep 5: Attention Decoder")
print("=" * 50)

# Create decoder
encoder_dim = sequence_encoder.output_size
decoder = AttentionDecoderV2(
    num_classes=config.NUM_CLASSES,
    encoder_dim=encoder_dim,
    embedding_dim=config.EMBEDDING_DIM,
    hidden_dim=config.HIDDEN_SIZE,
    attention_dim=config.ATTENTION_DIM
)

# Forward pass (without targets = teacher forcing disabled)
outputs, attention_weights = decoder(sequence_features, max_length=config.MAX_SEQ_LENGTH)

print(f"Input sequences: {sequence_features.shape}")
print(f"Output predictions: {outputs.shape}")
print(f"Expected: (B, max_length, num_classes) = (2, {config.MAX_SEQ_LENGTH}, {config.NUM_CLASSES})")
print(f"\nNumber of attention steps: {len(attention_weights)}")
print(f"Attention weights shape (first step): {attention_weights[0].shape}")

## Step 6: Complete ASTER Model

Now let's combine all components into the complete ASTER model.

In [ ]:
from model import ASTER, ASTERLoss

print("\nStep 6: Complete ASTER Model")
print("=" * 50)

# Create complete model
model = ASTER(config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test with random input
test_images = torch.randn(2, 3, 32, 100).to(device)

# Forward pass
model.eval()
with torch.no_grad():
    predictions, attention_weights, rectified, ctrl_points = model.predict(test_images)

print(f"\nInput shape: {test_images.shape}")
print(f"Rectified shape: {rectified.shape}")
print(f"Predictions shape: {predictions.shape}")

# Decode predictions
from utils import decode_prediction
for i in range(predictions.size(0)):
    text = decode_prediction(predictions[i], config.CHARACTERS)
    print(f"Sample {i+1}: '{text}'")

## Step 7: Understanding the Training Process

Let's examine the training loop and loss computation.

In [ ]:
# Create loss function
criterion = ASTERLoss(pad_token_idx=2)

print("\nStep 7: Loss Function")
print("=" * 50)

# Create dummy targets
targets = torch.randint(3, config.NUM_CLASSES, (2, config.MAX_SEQ_LENGTH)).to(device)

# Forward pass (with teacher forcing)
outputs, attention_weights, rectified, ctrl_points = model(
    test_images,
    targets=targets,
    teacher_forcing_ratio=0.5
)

# Calculate loss
loss, loss_dict = criterion(outputs, targets, ctrl_points, lambda_smooth=0.01)

print(f"Total Loss: {loss.item():.4f}")
print(f"Recognition Loss: {loss_dict['recognition_loss']:.4f}")
print(f"Smooth Loss: {loss_dict['smooth_loss']:.4f}")

print("\nLoss Components:")
print("- CrossEntropyLoss: Measures prediction accuracy")
print("- Smoothness Loss: Encourages smooth TPS transformations")

## Step 8: Dataset and Data Loading

ASTER can be trained on various datasets:
- **Synth90K**: 9 million synthetic word images
- **SynthText**: 8 million synthetic text images
- **IIIT5K**: Real scene text (5K images)
- **SVT**: Street View Text (349 images)
- **IC13/IC15**: ICDAR competition datasets

In [ ]:
from datasets import SyntheticTextGenerator, get_transforms
from torch.utils.data import DataLoader

print("\nStep 8: Dataset and Data Loading")
print("=" * 50)

# Create synthetic dataset for demonstration
train_transform = get_transforms(is_training=True, img_height=32, img_width=100)

synthetic_dataset = SyntheticTextGenerator(
    num_samples=1000,
    charset=config.CHARACTERS,
    img_height=config.IMG_HEIGHT,
    img_width=config.IMG_WIDTH,
    max_length=config.MAX_SEQ_LENGTH,
    transform=train_transform
)

print(f"Dataset size: {len(synthetic_dataset)}")

# Get a sample
sample = synthetic_dataset[0]
print(f"\nSample keys: {sample.keys()}")
print(f"Image shape: {sample['image'].shape}")
print(f"Label shape: {sample['label'].shape}")
print(f"Text: {sample['text']}")

# Create DataLoader
train_loader = DataLoader(
    synthetic_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

print(f"\nDataLoader created with {len(train_loader)} batches")

# Get a batch
batch = next(iter(train_loader))
print(f"Batch images shape: {batch['image'].shape}")
print(f"Batch labels shape: {batch['label'].shape}")

## Step 9: Training Loop

Let's see a simplified version of the training loop.

In [ ]:
from torch import optim

print("\nStep 9: Training Loop (Demo)")
print("=" * 50)

# Setup
model.train()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Run a few training steps
num_steps = 5
for step in range(num_steps):
    # Get batch
    batch = next(iter(train_loader))
    images = batch['image'].to(device)
    labels = batch['label'].to(device)
    
    # Forward
    outputs, attention_weights, rectified, ctrl_points = model(
        images,
        targets=labels,
        teacher_forcing_ratio=0.5
    )
    
    # Loss
    loss, loss_dict = criterion(outputs, labels, ctrl_points, lambda_smooth=0.01)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
    optimizer.step()
    
    print(f"Step {step+1}/{num_steps}: Loss = {loss.item():.4f}")

print("\nTraining steps completed!")
print("\nKey points:")
print("- Use teacher forcing (0.5 ratio) during training")
print("- Apply gradient clipping (max_norm=5)")
print("- Include smoothness loss for TPS")
print("- Use learning rate scheduler (ReduceLROnPlateau)")

## Step 10: Inference and Visualization

Let's visualize the model's outputs.

In [ ]:
print("\nStep 10: Inference and Visualization")
print("=" * 50)

# Generate a simple test image (white with black text area)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Create simple test image
test_img = torch.ones(1, 3, 32, 100)
test_img[:, :, 10:25, 20:80] = 0.0  # Black rectangle

# Show input
axes[0].imshow(test_img[0].permute(1, 2, 0).numpy())
axes[0].set_title('Input Image')
axes[0].axis('off')

# Run model
model.eval()
with torch.no_grad():
    predictions, attention_weights, rectified, ctrl_points = model.predict(test_img.to(device))
    
    # Show rectified
    rectified_np = rectified[0].cpu().permute(1, 2, 0).numpy()
    # Denormalize
    rectified_np = (rectified_np - rectified_np.min()) / (rectified_np.max() - rectified_np.min())
    axes[1].imshow(rectified_np)
    axes[1].set_title('Rectified Image')
    axes[1].axis('off')

plt.tight_layout()
plt.show()

# Visualize attention
fig, ax = plt.subplots(figsize=(10, 4))
avg_attention = torch.stack([aw[0] for aw in attention_weights[:10]]).mean(dim=0).cpu().numpy()
ax.bar(range(len(avg_attention)), avg_attention)
ax.set_xlabel('Position')
ax.set_ylabel('Attention Weight')
ax.set_title('Average Attention Distribution (first 10 steps)')
plt.show()

## Summary

### Key Features of ASTER:

1. **TPS Rectification**:
   - Automatically rectifies curved/distorted text
   - Uses 20 fiducial points
   - Differentiable transformation

2. **CNN Feature Extraction**:
   - ResNet-based architecture
   - Extracts robust visual features
   - Maintains spatial information

3. **Bidirectional LSTM**:
   - Models sequential dependencies
   - Processes in both directions
   - Context-aware representations

4. **Attention Decoder**:
   - Bahdanau attention mechanism
   - Dual LSTM architecture
   - Character-by-character prediction

### Training Tips:

- Start with synthetic data (Synth90K, SynthText)
- Use teacher forcing (0.5 ratio)
- Apply gradient clipping
- Fine-tune on real data (IIIT5K, SVT)
- Monitor rectification smoothness

### Files in the Implementation:

- `config.py` - Configuration classes
- `rectification.py` - TPS rectification network
- `feature_extractor.py` - CNN feature extraction
- `bidirectional_lstm.py` - Sequence modeling
- `attention_decoder.py` - Attention-based decoder
- `model.py` - Complete ASTER model
- `datasets.py` - Dataset classes
- `train.py` - Training script
- `inference.py` - Inference script
- `utils.py` - Utility functions

In [ ]:
print("\n" + "=" * 60)
print("ASTER Tutorial Complete!")
print("=" * 60)
print("\nTo train the model:")
print("  python train.py --dataset Synthetic --epochs 10")
print("\nTo run inference:")
print("  python inference.py --image path/to/image.jpg --checkpoint checkpoints/aster_best.pth")
print("\nGood luck with your OCR project!")